<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/Experiment_F_governance_fault_injection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import io
import os

print("📤 Please upload the diabetes_130US.csv file:")
uploaded = files.upload()
DATA_FILENAME  = "diabetes_130US.csv"
DATA_CACHE_DIR = "./data_cache"
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

# Get the uploaded file
for filename in uploaded.keys():
    file_data = uploaded[filename]
    print(f'✅ Uploaded: {filename} ({len(file_data)} bytes)')


    # Save to cache
    cache_path = os.path.join(DATA_CACHE_DIR, DATA_FILENAME)
    with open(cache_path, 'wb') as f:
        f.write(file_data)
    print(f'✅ Saved to cache: {cache_path}')

    # Verify the file was saved
    if os.path.exists(cache_path):
        file_size = os.path.getsize(cache_path)
        print(f'✅ Verified: {cache_path} exists ({file_size} bytes)')
    else:
        print(f'❌ Error: File was not saved properly')

📤 Please upload the diabetes_130US.csv file:


Saving diabetes_130US.csv to diabetes_130US.csv
✅ Uploaded: diabetes_130US.csv (19159383 bytes)
✅ Saved to cache: ./data_cache/diabetes_130US.csv
✅ Verified: ./data_cache/diabetes_130US.csv exists (19159383 bytes)


In [4]:
# ======================================================================================
# TADP EXPERIMENT F v3 — MANDATORY-MINIMUM FAULT INJECTION + REVIEW SCORE DIAGNOSTIC
# ======================================================================================
# Purpose
# -------
# Validate the revised TADP admission policy at the governance layer only:
#
#   1) Mandatory Governance Gate:
#      each policy-mandatory factor must be present, finite, and meet its
#      factor-specific minimum. Failure -> immediate Reject; no compensation.
#   2) Dimension floor: every averaged dimension >= 2.5/5.
#   3) HPS < 3.0 -> Reject.
#   4) HPS >= 3.5 -> Accept.
#   5) 3.0 <= HPS < 3.5 -> automated review using Review Score.
#   6) Review Score >= 3.25 -> Accept after review; otherwise Reject.
#
# Experiment F v3 extends v2 by testing INSUFFICIENT mandatory evidence, not only
# absent/zero evidence. Each mandatory factor is injected at four failure severities:
# MISSING, ZERO, ONE, and JUST_BELOW_MINIMUM. A separate boundary self-test confirms
# that a factor exactly at its policy minimum passes the Mandatory Gate.
#
# This experiment performs NO model training and uses NO test data.
# ======================================================================================

import os
import json
import math
import time
import hashlib
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd


# ======================================================================================
# VERSION / OUTPUT
# ======================================================================================

EXPERIMENT_VERSION = "TADP-RQ16-GOVFAULT-v20.0-MANDATORYMIN-REVIEWSCORE325-K20-20SEEDS"
OUTPUT_ROOT = Path("/content/TADP_Experiment_F_v3_Governance_Fault_Injection")
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================================
# FROZEN TADP POLICY
# ======================================================================================

HPS_REJECT_THRESHOLD = 3.0
HPS_ACCEPT_THRESHOLD = 3.5
DIMENSION_MIN_FLOOR = 2.5
MAX_FACTOR_SCORE = 5.0
REVIEW_ACCEPT_THRESHOLD = (HPS_REJECT_THRESHOLD + HPS_ACCEPT_THRESHOLD) / 2.0  # 3.25

WEIGHTS_PSCORE = {
    "dim1": 0.25,
    "dim2": 0.15,
    "dim3": 0.10,
    "dim4": 0.10,
    "dim5": 0.30,
    "dim6": 0.10,
}

FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

FACTOR_ADEQUACY_MIN = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

# Exactly the six non-negotiable requirements frozen for this healthcare policy.
MANDATORY_FACTORS = {
    "dim1": [
        "data_controller",
        "data_collection_lineage",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "sensitivity_classification",
    ],
    "dim6": [
        "license_terms",
    ],
}

# Compensable factors used only to resolve the HPS review band.
REVIEW_FACTORS = {
    "dim1": [
        "source_reputation",
        "data_objective",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "geo_restrictions",
        "audits",
    ],
    "dim6": [
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

assert abs(sum(WEIGHTS_PSCORE.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert abs(REVIEW_ACCEPT_THRESHOLD - 3.25) < 1e-12


# ======================================================================================
# EXPERIMENT CONFIGURATION
# ======================================================================================

K_CLIENTS = 20
CLIENT_IDS = [f"C{i+1:02d}" for i in range(K_CLIENTS)]

PREVALENCE_LEVELS = [0.10, 0.25, 0.50]
INJECTION_SEEDS = [1042 + 1000 * i for i in range(20)]

MANDATORY_FAILURE_SEVERITIES = [
    "MISSING",
    "ZERO",
    "ONE",
    "JUST_BELOW_MINIMUM",
]

FAULT_CATALOG = [
    # Mandatory faults: four severities are tested for each.
    {
        "fault_id": "M01",
        "fault_name": "Insufficient data-controller evidence",
        "dimension": "dim1",
        "factor": "data_controller",
        "mandatory": True,
        "policy_meaning": "Required controller evidence must meet its configured minimum.",
    },
    {
        "fault_id": "M02",
        "fault_name": "Insufficient acquisition-lineage evidence",
        "dimension": "dim1",
        "factor": "data_collection_lineage",
        "mandatory": True,
        "policy_meaning": "Required acquisition lineage must meet its configured minimum.",
    },
    {
        "fault_id": "M03",
        "fault_name": "Insufficient applicable-regulatory evidence",
        "dimension": "dim5",
        "factor": "regulation_coverage",
        "mandatory": True,
        "policy_meaning": "Applicable regulatory evidence must meet its configured minimum.",
    },
    {
        "fault_id": "M04",
        "fault_name": "Insufficient required consent/authorization evidence",
        "dimension": "dim5",
        "factor": "consent_ethics",
        "mandatory": True,
        "policy_meaning": "Required consent/authorization evidence must meet its configured minimum.",
    },
    {
        "fault_id": "M05",
        "fault_name": "Insufficient sensitivity classification",
        "dimension": "dim5",
        "factor": "sensitivity_classification",
        "mandatory": True,
        "policy_meaning": "Required sensitivity classification must meet its configured minimum.",
    },
    {
        "fault_id": "M06",
        "fault_name": "Insufficient required usage-rights evidence",
        "dimension": "dim6",
        "factor": "license_terms",
        "mandatory": True,
        "policy_meaning": "Required licence/usage-rights evidence must meet its configured minimum.",
    },

    # Graded faults: score 0 is injected, but these are NOT universal hard vetoes.
    {
        "fault_id": "G01",
        "fault_name": "Missing data dictionary",
        "dimension": "dim3",
        "factor": "data_dictionary",
        "mandatory": False,
        "policy_meaning": "Compensable documentation deficiency.",
    },
    {
        "fault_id": "G02",
        "fault_name": "Missing collection protocol",
        "dimension": "dim3",
        "factor": "collection_protocol",
        "mandatory": False,
        "policy_meaning": "Compensable documentation deficiency.",
    },
    {
        "fault_id": "G03",
        "fault_name": "Missing freshness evidence",
        "dimension": "dim4",
        "factor": "data_freshness",
        "mandatory": False,
        "policy_meaning": "Compensable timeliness deficiency.",
    },
    {
        "fault_id": "G04",
        "fault_name": "Missing audit evidence",
        "dimension": "dim5",
        "factor": "audits",
        "mandatory": False,
        "policy_meaning": "Compensable audit-maturity deficiency.",
    },
    {
        "fault_id": "G05",
        "fault_name": "Missing ethical-review evidence",
        "dimension": "dim6",
        "factor": "ethical_reviews",
        "mandatory": False,
        "policy_meaning": "Compensable contextual-governance deficiency.",
    },
    {
        "fault_id": "G06",
        "fault_name": "Weak/missing user-agreement evidence",
        "dimension": "dim6",
        "factor": "user_agreements",
        "mandatory": False,
        "policy_meaning": "Compensable usage-context deficiency.",
    },
]


# ======================================================================================
# POLICY FUNCTIONS
# ======================================================================================

def deep_copy_factors(factors: Dict[str, Dict[str, float]]) -> Dict[str, Dict[str, float]]:
    return {
        dim: {factor: float(value) for factor, value in vals.items()}
        for dim, vals in factors.items()
    }


def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    out = {}
    for dim in FACTOR_NAMES:
        values = []
        for factor in FACTOR_NAMES[dim]:
            value = factors.get(dim, {}).get(factor, np.nan)
            value = float(value)
            # For dimension/HPS calculation, missing/non-finite evidence contributes
            # 0.0. Mandatory missing evidence is already fail-closed by Gate 1.
            values.append(value if np.isfinite(value) else 0.0)
        out[dim] = float(np.mean(values))
    return out


def compute_hps(dimensions: Dict[str, float]) -> float:
    return float(
        sum(WEIGHTS_PSCORE[d] * dimensions[d] for d in WEIGHTS_PSCORE)
    )


def mandatory_gate_audit(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    scores = {}
    failures = []
    missing = []
    below = []

    for dim, factor_list in MANDATORY_FACTORS.items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            minimum = float(FACTOR_ADEQUACY_MIN[dim][factor])

            if factor not in factors.get(dim, {}):
                missing.append(key)
                failures.append(f"{key}:MISSING")
                continue

            value = float(factors[dim][factor])
            if not np.isfinite(value):
                missing.append(key)
                failures.append(f"{key}:MISSING")
                continue

            scores[key] = value
            if value < minimum:
                below.append(f"{key}:{value:.1f}<{minimum:.1f}")
                failures.append(f"{key}:{value:.1f}<{minimum:.1f}")

    return {
        "passed": len(failures) == 0,
        "scores": scores,
        "failures": failures,
        "missing": missing,
        "below_minimum": below,
    }


def compute_review_score(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    values = []
    scores = {}
    missing = []

    for dim, factor_list in REVIEW_FACTORS.items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            value = factors.get(dim, {}).get(factor, np.nan)
            value = float(value)
            if not np.isfinite(value):
                missing.append(key)
                value = 0.0
            value = float(np.clip(value, 0.0, MAX_FACTOR_SCORE))
            scores[key] = value
            values.append(value)

    score = float(np.mean(values))
    return {
        "review_score_0_5": score,
        "review_acceptance_threshold_0_5": float(REVIEW_ACCEPT_THRESHOLD),
        "review_score_margin": float(score - REVIEW_ACCEPT_THRESHOLD),
        "scores": scores,
        "missing": missing,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    mandatory = mandatory_gate_audit(factors)
    dimensions = dimension_scores_from_factors(factors)
    hps = compute_hps(dimensions)
    review = compute_review_score(factors)

    dimension_failures = [
        dim for dim, value in dimensions.items()
        if value < DIMENSION_MIN_FLOOR
    ]

    base = {
        "mandatory_gate_pass": bool(mandatory["passed"]),
        "mandatory_failures": ";".join(mandatory["failures"]),
        "dimensions": dimensions,
        "hps": float(hps),
        "review_score_0_5": float(review["review_score_0_5"]),
        "review_acceptance_threshold_0_5": float(REVIEW_ACCEPT_THRESHOLD),
        "review_score_margin": float(review["review_score_margin"]),
        "dimension_floor_failures": ";".join(dimension_failures),
    }

    if not mandatory["passed"]:
        return {
            **base,
            "decision_path": "MANDATORY_GOVERNANCE_GATE",
            "final_action": "REJECT",
            "reason": "Mandatory requirement failed: " + ";".join(mandatory["failures"]),
        }

    if dimension_failures:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "final_action": "REJECT",
            "reason": (
                f"At least one dimension < {DIMENSION_MIN_FLOOR:.1f}: "
                + ";".join(
                    f"{d}={dimensions[d]:.3f}" for d in dimension_failures
                )
            ),
        }

    if hps < HPS_REJECT_THRESHOLD:
        return {
            **base,
            "decision_path": "LOW_HPS",
            "final_action": "REJECT",
            "reason": f"HPS {hps:.3f} < {HPS_REJECT_THRESHOLD:.3f}",
        }

    if hps >= HPS_ACCEPT_THRESHOLD:
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "reason": f"HPS {hps:.3f} >= {HPS_ACCEPT_THRESHOLD:.3f}",
        }

    if review["review_score_0_5"] >= REVIEW_ACCEPT_THRESHOLD:
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "final_action": "ACCEPT",
            "reason": (
                f"Review Score {review['review_score_0_5']:.3f} >= "
                f"{REVIEW_ACCEPT_THRESHOLD:.3f}"
            ),
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "final_action": "REJECT",
        "reason": (
            f"Review Score {review['review_score_0_5']:.3f} < "
            f"{REVIEW_ACCEPT_THRESHOLD:.3f}"
        ),
    }


# ======================================================================================
# CLEAN CONTROL PROFILES
# ======================================================================================

def make_direct_strong_profile(variant: int) -> Dict[str, Dict[str, float]]:
    factors = {}
    for dim in FACTOR_NAMES:
        factors[dim] = {}
        for j, factor in enumerate(FACTOR_NAMES[dim]):
            base = max(4.0, float(FACTOR_ADEQUACY_MIN[dim][factor]))
            bonus = 1.0 if ((j + variant + len(dim)) % 4 == 0) else 0.0
            factors[dim][factor] = float(min(5.0, base + bonus))
    return factors


def make_review_pass_profile(variant: int) -> Dict[str, Dict[str, float]]:
    # Clean borderline profile:
    #   Mandatory factors meet their exact minima.
    #   DQ = 3.0.
    #   Documentation Practice review factors = 4.
    #   Other review factors = 3.
    # This produces HPS around 3.225 and Review Score around 3.286.
    factors = {
        dim: {factor: 3.0 for factor in FACTOR_NAMES[dim]}
        for dim in FACTOR_NAMES
    }

    for dim, factor_list in MANDATORY_FACTORS.items():
        for factor in factor_list:
            factors[dim][factor] = float(FACTOR_ADEQUACY_MIN[dim][factor])

    for factor in FACTOR_NAMES["dim3"]:
        factors["dim3"][factor] = 4.0

    if variant % 2 == 1:
        # Preserve the same aggregate Review Score while varying the evidence pattern.
        factors["dim3"]["definition_updates"] = 3.0
        factors["dim4"]["scheduled_refresh"] = 4.0

    return factors


def build_clean_profiles() -> Dict[str, Dict[str, Dict[str, float]]]:
    profiles = {}
    for i, cid in enumerate(CLIENT_IDS):
        if i < 10:
            profiles[cid] = make_direct_strong_profile(i)
        else:
            profiles[cid] = make_review_pass_profile(i - 10)

    # Fail closed if any clean control would be rejected before fault injection.
    decisions = {cid: tadp_decision(p) for cid, p in profiles.items()}
    rejected = [
        cid for cid, d in decisions.items()
        if d["final_action"] != "ACCEPT"
    ]
    if rejected:
        raise RuntimeError(
            f"Clean-control construction failed; rejected controls: {rejected}"
        )
    return profiles


# ======================================================================================
# FAULT INJECTION
# ======================================================================================

def injected_value_for_fault(
    fault: Dict[str, Any],
    severity: str,
) -> float:
    dim = fault["dimension"]
    factor = fault["factor"]

    if not fault["mandatory"]:
        if severity != "ZERO":
            raise ValueError("Graded faults currently use severity='ZERO' only.")
        return 0.0

    minimum = float(FACTOR_ADEQUACY_MIN[dim][factor])

    if severity == "MISSING":
        return float("nan")
    if severity == "ZERO":
        return 0.0
    if severity == "ONE":
        return 1.0
    if severity == "JUST_BELOW_MINIMUM":
        return float(max(0.0, minimum - 1.0))

    raise ValueError(f"Unknown severity: {severity}")


def inject_fault(
    base_profiles: Dict[str, Dict[str, Dict[str, float]]],
    fault: Dict[str, Any],
    severity: str,
    prevalence: float,
    seed: int,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], List[str]]:
    rng = np.random.default_rng(int(seed))
    n_faulted = max(1, int(round(float(prevalence) * len(CLIENT_IDS))))
    faulted_clients = sorted(
        rng.choice(CLIENT_IDS, size=n_faulted, replace=False).tolist()
    )

    out = {
        cid: deep_copy_factors(profile)
        for cid, profile in base_profiles.items()
    }

    value = injected_value_for_fault(fault, severity)
    dim = fault["dimension"]
    factor = fault["factor"]

    for cid in faulted_clients:
        out[cid][dim][factor] = float(value)

    return out, faulted_clients


# ======================================================================================
# BOUNDARY SELF-TESTS
# ======================================================================================

def mandatory_boundary_self_test(
    clean_profiles: Dict[str, Dict[str, Dict[str, float]]]
) -> pd.DataFrame:
    rows = []
    reference = make_direct_strong_profile(999)

    for fault in [f for f in FAULT_CATALOG if f["mandatory"]]:
        dim = fault["dimension"]
        factor = fault["factor"]
        minimum = float(FACTOR_ADEQUACY_MIN[dim][factor])

        at_min = deep_copy_factors(reference)
        at_min[dim][factor] = minimum
        d_at = tadp_decision(at_min)

        below = deep_copy_factors(reference)
        below[dim][factor] = max(0.0, minimum - 1.0)
        d_below = tadp_decision(below)

        rows.append({
            "fault_id": fault["fault_id"],
            "factor": f"{dim}.{factor}",
            "mandatory_minimum": minimum,
            "at_min_gate_pass": bool(d_at["mandatory_gate_pass"]),
            "at_min_final_action": d_at["final_action"],
            "below_min_gate_pass": bool(d_below["mandatory_gate_pass"]),
            "below_min_final_action": d_below["final_action"],
            "boundary_test_pass": bool(
                d_at["mandatory_gate_pass"]
                and not d_below["mandatory_gate_pass"]
                and d_below["final_action"] == "REJECT"
            ),
        })

    out = pd.DataFrame(rows)
    if not out["boundary_test_pass"].all():
        raise RuntimeError(
            "Mandatory-boundary self-test failed:\n"
            + out.to_string(index=False)
        )
    return out


def clean_branch_self_test(
    clean_profiles: Dict[str, Dict[str, Dict[str, float]]]
) -> pd.DataFrame:
    rows = []
    for cid, profile in clean_profiles.items():
        d = tadp_decision(profile)
        rows.append({
            "client": cid,
            "hps": d["hps"],
            "review_score_0_5": d["review_score_0_5"],
            "mandatory_gate_pass": d["mandatory_gate_pass"],
            "decision_path": d["decision_path"],
            "final_action": d["final_action"],
        })

    out = pd.DataFrame(rows)
    if not out["final_action"].eq("ACCEPT").all():
        raise RuntimeError("Clean branch self-test contains rejected clients.")
    if not out["decision_path"].eq("HPS_REVIEW_BAND").any():
        raise RuntimeError(
            "Clean controls do not exercise the HPS review band."
        )
    return out


# ======================================================================================
# MAIN EXPERIMENT
# ======================================================================================

def main():
    start = time.perf_counter()

    print("=" * 108)
    print("TADP EXPERIMENT F v3 — MANDATORY-MINIMUM FAULT INJECTION + REVIEW SCORE")
    print("=" * 108)
    print(f"Version: {EXPERIMENT_VERSION}")
    print(f"Clients: K={K_CLIENTS}")
    print(f"Prevalence levels: {PREVALENCE_LEVELS}")
    print(f"Injection seeds: {len(INJECTION_SEEDS)}")
    print(
        f"Review Acceptance Threshold: {REVIEW_ACCEPT_THRESHOLD:.2f}/5 "
        f"(midpoint of {HPS_REJECT_THRESHOLD:.1f} and {HPS_ACCEPT_THRESHOLD:.1f})"
    )
    print("No model training; governance-only diagnostic.")

    clean_profiles = build_clean_profiles()

    boundary = mandatory_boundary_self_test(clean_profiles)
    boundary.to_csv(
        OUTPUT_DIR / "mandatory_minimum_boundary_self_test.csv",
        index=False,
    )

    clean_branches = clean_branch_self_test(clean_profiles)
    clean_branches.to_csv(
        OUTPUT_DIR / "clean_control_branch_audit.csv",
        index=False,
    )

    # Frozen policy configuration.
    policy = {
        "experiment_version": EXPERIMENT_VERSION,
        "K_clients": K_CLIENTS,
        "prevalence_levels": PREVALENCE_LEVELS,
        "injection_seeds": INJECTION_SEEDS,
        "hps_reject_threshold": HPS_REJECT_THRESHOLD,
        "hps_accept_threshold": HPS_ACCEPT_THRESHOLD,
        "dimension_min_floor": DIMENSION_MIN_FLOOR,
        "review_acceptance_threshold_0_5": REVIEW_ACCEPT_THRESHOLD,
        "review_threshold_derivation": "(3.0 + 3.5) / 2 = 3.25",
        "mandatory_factors": MANDATORY_FACTORS,
        "mandatory_minima": {
            dim: {
                factor: FACTOR_ADEQUACY_MIN[dim][factor]
                for factor in factors
            }
            for dim, factors in MANDATORY_FACTORS.items()
        },
        "review_factors": REVIEW_FACTORS,
        "mandatory_failure_severities": MANDATORY_FAILURE_SEVERITIES,
        "fault_catalog": FAULT_CATALOG,
        "model_training": False,
        "test_data_used": False,
    }
    (OUTPUT_DIR / "frozen_policy.json").write_text(
        json.dumps(policy, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    condition_rows = []
    client_rows = []

    variants = []
    for fault in FAULT_CATALOG:
        if fault["mandatory"]:
            for severity in MANDATORY_FAILURE_SEVERITIES:
                variants.append((fault, severity))
        else:
            variants.append((fault, "ZERO"))

    total_conditions = (
        len(variants)
        * len(PREVALENCE_LEVELS)
        * len(INJECTION_SEEDS)
    )

    counter = 0

    for fault, severity in variants:
        for prevalence in PREVALENCE_LEVELS:
            for seed in INJECTION_SEEDS:
                counter += 1

                injected_profiles, faulted_clients = inject_fault(
                    clean_profiles,
                    fault,
                    severity,
                    prevalence,
                    seed,
                )
                faulted_set = set(faulted_clients)

                decisions = {
                    cid: tadp_decision(profile)
                    for cid, profile in injected_profiles.items()
                }

                faulted_decisions = [
                    decisions[cid] for cid in faulted_clients
                ]
                clean_ids = [
                    cid for cid in CLIENT_IDS if cid not in faulted_set
                ]
                clean_decisions = [
                    decisions[cid] for cid in clean_ids
                ]

                fault_reject_rate = float(np.mean([
                    d["final_action"] == "REJECT"
                    for d in faulted_decisions
                ]))

                clean_frr = float(np.mean([
                    d["final_action"] == "REJECT"
                    for d in clean_decisions
                ])) if clean_decisions else 0.0

                mandatory_recall = np.nan
                mandatory_gate_recall = np.nan

                if fault["mandatory"]:
                    mandatory_recall = fault_reject_rate
                    mandatory_gate_recall = float(np.mean([
                        d["decision_path"] == "MANDATORY_GOVERNANCE_GATE"
                        and d["final_action"] == "REJECT"
                        for d in faulted_decisions
                    ]))

                condition_rows.append({
                    "fault_id": fault["fault_id"],
                    "fault_name": fault["fault_name"],
                    "fault_is_mandatory": bool(fault["mandatory"]),
                    "dimension": fault["dimension"],
                    "factor": fault["factor"],
                    "severity": severity,
                    "injected_value": injected_value_for_fault(fault, severity),
                    "prevalence": float(prevalence),
                    "seed": int(seed),
                    "n_faulted": int(len(faulted_clients)),
                    "faulted_clients": ";".join(faulted_clients),
                    "fault_rejection_rate": fault_reject_rate,
                    "mandatory_recall": mandatory_recall,
                    "mandatory_gate_recall": mandatory_gate_recall,
                    "clean_false_rejection_rate": clean_frr,
                    "faulted_mean_hps": float(np.mean([
                        d["hps"] for d in faulted_decisions
                    ])),
                    "faulted_mean_review_score": float(np.mean([
                        d["review_score_0_5"] for d in faulted_decisions
                    ])),
                })

                for cid in CLIENT_IDS:
                    d = decisions[cid]
                    client_rows.append({
                        "fault_id": fault["fault_id"],
                        "fault_name": fault["fault_name"],
                        "fault_is_mandatory": bool(fault["mandatory"]),
                        "dimension": fault["dimension"],
                        "factor": fault["factor"],
                        "severity": severity,
                        "prevalence": float(prevalence),
                        "seed": int(seed),
                        "client": cid,
                        "is_faulted": cid in faulted_set,
                        "mandatory_gate_pass": bool(d["mandatory_gate_pass"]),
                        "mandatory_failures": d["mandatory_failures"],
                        "hps": float(d["hps"]),
                        "review_score_0_5": float(d["review_score_0_5"]),
                        "review_acceptance_threshold_0_5": float(
                            d["review_acceptance_threshold_0_5"]
                        ),
                        "dimension_floor_failures": d["dimension_floor_failures"],
                        "decision_path": d["decision_path"],
                        "final_action": d["final_action"],
                        "reason": d["reason"],
                    })

                if counter == 1 or counter % 120 == 0 or counter == total_conditions:
                    mand_txt = (
                        f"{mandatory_gate_recall:.3f}"
                        if np.isfinite(mandatory_gate_recall)
                        else "n/a"
                    )
                    print(
                        f"[{counter:4d}/{total_conditions}] "
                        f"{fault['fault_id']} | severity={severity} | "
                        f"mandatory={fault['mandatory']} | prevalence={prevalence:.2f} | "
                        f"seed={seed} | fault_reject={fault_reject_rate:.3f} | "
                        f"mandatory_gate_recall={mand_txt} | clean_FRR={clean_frr:.3f}"
                    )

    conditions = pd.DataFrame(condition_rows)
    clients = pd.DataFrame(client_rows)

    conditions.to_csv(
        OUTPUT_DIR / "condition_level_results.csv",
        index=False,
    )
    clients.to_csv(
        OUTPUT_DIR / "client_level_results.csv",
        index=False,
    )

    # Fault/severity summary.
    summary = (
        conditions
        .groupby(
            [
                "fault_id",
                "fault_name",
                "fault_is_mandatory",
                "dimension",
                "factor",
                "severity",
            ],
            as_index=False,
        )
        .agg(
            fault_rejection_rate_mean=("fault_rejection_rate", "mean"),
            fault_rejection_rate_min=("fault_rejection_rate", "min"),
            fault_rejection_rate_max=("fault_rejection_rate", "max"),
            mandatory_gate_recall_mean=("mandatory_gate_recall", "mean"),
            mandatory_gate_recall_min=("mandatory_gate_recall", "min"),
            clean_false_rejection_rate_mean=("clean_false_rejection_rate", "mean"),
            clean_false_rejection_rate_max=("clean_false_rejection_rate", "max"),
            faulted_mean_hps=("faulted_mean_hps", "mean"),
            faulted_mean_review_score=("faulted_mean_review_score", "mean"),
        )
    )

    def classify(row):
        if bool(row["fault_is_mandatory"]):
            if (
                np.isfinite(row["mandatory_gate_recall_min"])
                and row["mandatory_gate_recall_min"] >= 1.0 - 1e-12
            ):
                return "CONSISTENTLY_BLOCKED_BY_MANDATORY_GATE"
            return "MANDATORY_GATE_FAILURE"
        return "GRADED_POLICY_RESPONSE_NOT_HARD_VETO"

    summary["coverage_class"] = summary.apply(classify, axis=1)
    summary.to_csv(
        OUTPUT_DIR / "fault_severity_policy_coverage_summary.csv",
        index=False,
    )

    # Compact mandatory-factor summary across every below-minimum severity.
    mandatory_summary = (
        conditions[conditions["fault_is_mandatory"]]
        .groupby(
            ["fault_id", "fault_name", "dimension", "factor"],
            as_index=False,
        )
        .agg(
            mandatory_gate_recall_mean=("mandatory_gate_recall", "mean"),
            mandatory_gate_recall_min=("mandatory_gate_recall", "min"),
            final_rejection_recall_mean=("mandatory_recall", "mean"),
            final_rejection_recall_min=("mandatory_recall", "min"),
            clean_false_rejection_rate_max=("clean_false_rejection_rate", "max"),
        )
    )
    mandatory_summary.to_csv(
        OUTPUT_DIR / "mandatory_factor_summary.csv",
        index=False,
    )

    # Decision-path summary for graded faults.
    graded_clients = clients[
        (~clients["fault_is_mandatory"])
        & clients["is_faulted"]
    ].copy()
    graded_path = (
        graded_clients
        .groupby(
            ["fault_id", "fault_name", "decision_path", "final_action"],
            as_index=False,
        )
        .size()
        .rename(columns={"size": "client_decisions"})
    )
    graded_path.to_csv(
        OUTPUT_DIR / "graded_fault_decision_path_summary.csv",
        index=False,
    )

    # Fail-closed validation of the experiment itself.
    min_mandatory_gate_recall = float(
        mandatory_summary["mandatory_gate_recall_min"].min()
    )
    max_clean_frr = float(
        conditions["clean_false_rejection_rate"].max()
    )
    boundary_pass = bool(boundary["boundary_test_pass"].all())

    validation = {
        "mandatory_boundary_self_test_pass": boundary_pass,
        "minimum_mandatory_gate_recall_across_all_factors_severities_prevalences_seeds":
            min_mandatory_gate_recall,
        "maximum_clean_false_rejection_rate": max_clean_frr,
        "review_threshold_is_midpoint_3_25": bool(
            abs(REVIEW_ACCEPT_THRESHOLD - 3.25) < 1e-12
        ),
        "mandatory_gate_self_test_pass": bool(
            boundary_pass
            and min_mandatory_gate_recall >= 1.0 - 1e-12
            and max_clean_frr <= 1e-12
        ),
    }
    (OUTPUT_DIR / "experiment_validation.json").write_text(
        json.dumps(validation, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    if not validation["mandatory_gate_self_test_pass"]:
        raise RuntimeError(
            "Experiment F v3 fail-closed validation FAILED:\n"
            + json.dumps(validation, indent=2)
        )

    # Result manifest.
    manifest_rows = []
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            manifest_rows.append({
                "relative_path": str(p.relative_to(OUTPUT_DIR)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest_rows).to_csv(
        OUTPUT_DIR / "RESULTS_FILE_MANIFEST.csv",
        index=False,
    )

    elapsed = time.perf_counter() - start

    print("\n" + "=" * 108)
    print("TADP EXPERIMENT F v3 COMPLETE")
    print("=" * 108)
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Conditions: {len(conditions):,}")
    print(f"Client-level evaluations: {len(clients):,}")
    print(f"Elapsed seconds: {elapsed:.3f}")
    print(f"Review Acceptance Threshold: {REVIEW_ACCEPT_THRESHOLD:.2f}/5")
    print("\nMandatory factor summary:")
    print(mandatory_summary.to_string(index=False))
    print("\nBoundary self-test:")
    print(boundary.to_string(index=False))
    print(
        f"\nMinimum mandatory-gate recall = {min_mandatory_gate_recall:.3f}"
    )
    print(f"Maximum clean false-rejection rate = {max_clean_frr:.3f}")
    print("Mandatory-gate self-test: PASS")

    zip_path = OUTPUT_ROOT / f"{EXPERIMENT_VERSION}.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as zf:
        for p in sorted(OUTPUT_DIR.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))

    print(f"\nZIP package: {zip_path}")

    try:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    except Exception:
        pass

    return OUTPUT_DIR, zip_path


if __name__ == "__main__":
    main()


TADP EXPERIMENT F v3 — MANDATORY-MINIMUM FAULT INJECTION + REVIEW SCORE
Version: TADP-RQ16-GOVFAULT-v20.0-MANDATORYMIN-REVIEWSCORE325-K20-20SEEDS
Clients: K=20
Prevalence levels: [0.1, 0.25, 0.5]
Injection seeds: 20
Review Acceptance Threshold: 3.25/5 (midpoint of 3.0 and 3.5)
No model training; governance-only diagnostic.
[   1/1800] M01 | severity=MISSING | mandatory=True | prevalence=0.10 | seed=1042 | fault_reject=1.000 | mandatory_gate_recall=1.000 | clean_FRR=0.000
[ 120/1800] M01 | severity=ZERO | mandatory=True | prevalence=0.50 | seed=20042 | fault_reject=1.000 | mandatory_gate_recall=1.000 | clean_FRR=0.000
[ 240/1800] M01 | severity=JUST_BELOW_MINIMUM | mandatory=True | prevalence=0.50 | seed=20042 | fault_reject=1.000 | mandatory_gate_recall=1.000 | clean_FRR=0.000
[ 360/1800] M02 | severity=ZERO | mandatory=True | prevalence=0.50 | seed=20042 | fault_reject=1.000 | mandatory_gate_recall=1.000 | clean_FRR=0.000
[ 480/1800] M02 | severity=JUST_BELOW_MINIMUM | mandatory=True 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>